# 01 — Dataset Exploration

**Goal.** Inventory the SignBD-Word corpus and profile the video containers before any processing.

**Approach**
- Walk the dataset tree, find the class-folder depth, and count clips per class. Check for train/test/val folders and signer-like folder names.
- Read container metadata (frame count, FPS, resolution, duration) for every video with OpenCV.
- Summarise and plot the class balance and the duration / FPS / frame-count / resolution distributions.

**Findings that shaped the pipeline**
- 6,000 clips, 200 classes × 30 clips, 16 signers (p2 and p3 have 200 clips each; the rest have 400).
- FPS spans ~26–60 in two regimes (~30 and ~60) and duration spans 0.66–6.8 s → sequences are variable-length and a frame index is not a common unit of time. Later stages handle this explicitly (09: physical-time features; 08/11/12: native frames with a length cap).

**Outputs.** `folder_inventory.csv`, `class_folder_summary.csv`, `split_folder_summary.csv`, `signer_folder_summary.csv`, `signbd_metadata.csv`, `statistical_summary.csv`.

In [ ]:
# 1) Imports

import os
import re
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
# 2) Set dataset path and auto-detect the real dataset folder

DATASET_ROOT = Path("/kaggle/input/datasets/shahriar55/signbd-dataset/SignBD-Word")

# If SignBD-Word exists inside the root, use that
if (DATASET_ROOT / "SignBD-Word").exists():
    DATASET_PATH = DATASET_ROOT / "SignBD-Word"
else:
    DATASET_PATH = DATASET_ROOT

print("DATASET_ROOT :", DATASET_ROOT)
print("DATASET_PATH :", DATASET_PATH)

In [ ]:
# 3) Show only top-level folders

top_level_folders = sorted([p.name for p in DATASET_PATH.iterdir() if p.is_dir()])

print("Top-level folders:")
for folder in top_level_folders:
    print(folder)

print("\nTotal top-level folders:", len(top_level_folders))

In [ ]:
# 4) Print folder tree only, not files

def print_dir_tree(path, max_depth=None, level=0):
    path = Path(path)

    print("    " * level + f"{path.name}/")

    if max_depth is not None and level >= max_depth:
        return

    subdirs = sorted([p for p in path.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
    for subdir in subdirs:
        print_dir_tree(subdir, max_depth=max_depth, level=level + 1)

print_dir_tree(DATASET_PATH, max_depth=5)

In [ ]:
# 5) Build a folder inventory table

folder_rows = []

def scan_folders(path):
    path = Path(path)
    subdirs = sorted([p for p in path.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

    folder_rows.append({
        "folder": str(path),
        "name": path.name,
        "depth": len(path.relative_to(DATASET_PATH).parts),
        "parent": str(path.parent) if path != DATASET_PATH else "",
        "subfolders": len(subdirs),
        "is_leaf": len(subdirs) == 0
    })

    for subdir in subdirs:
        scan_folders(subdir)

scan_folders(DATASET_PATH)

folder_df = pd.DataFrame(folder_rows)
folder_df = folder_df.sort_values(["depth", "folder"]).reset_index(drop=True)

folder_df.head(20)

In [ ]:
# 6) Save the folder inventory

folder_df.to_csv("/kaggle/working/folder_inventory.csv", index=False)
print("Saved: /kaggle/working/folder_inventory.csv")

In [ ]:
# 7) Folder-depth summary

depth_summary = (
    folder_df.groupby("depth")
    .agg(
        total_folders=("folder", "count"),
        leaf_folders=("is_leaf", "sum")
    )
    .reset_index()
)

depth_summary

In [ ]:
# 8) Plot folder distribution by depth

plt.figure(figsize=(10, 4))
plt.bar(depth_summary["depth"].astype(str), depth_summary["total_folders"])
plt.xlabel("Folder depth")
plt.ylabel("Number of folders")
plt.title("Folder distribution by depth")
plt.tight_layout()
plt.show()

In [ ]:
# 9) Inspect folders at each depth

for d in sorted(folder_df["depth"].unique()):
    print(f"\n=== Depth {d} ===")
    display(folder_df[folder_df["depth"] == d][["folder", "subfolders", "is_leaf"]].head(30))

In [ ]:
# 10) Choose the class-folder depth

CLASS_DEPTH = 1   # change this after checking the folder tree

In [ ]:
# 11) Extract class folders

class_folder_df = folder_df[folder_df["depth"] == CLASS_DEPTH].copy()
class_folder_df = class_folder_df.sort_values("name").reset_index(drop=True)

class_folder_df[["name", "folder", "subfolders", "is_leaf"]].head(50)

In [ ]:
# 12) Count files inside class folders

def count_files_in_folder(folder_path):
    folder_path = Path(folder_path)
    return sum(1 for p in folder_path.iterdir() if p.is_file())

class_folder_df["file_count"] = class_folder_df["folder"].apply(count_files_in_folder)

class_folder_df[["name", "folder", "file_count", "subfolders", "is_leaf"]].head(50)

In [ ]:
# 13) Class distribution plot

plt.figure(figsize=(20, 6))
plt.bar(class_folder_df["name"], class_folder_df["file_count"])
plt.xticks(rotation=90)
plt.xlabel("Class")
plt.ylabel("File count")
plt.title("Class distribution")
plt.tight_layout()
plt.show()

In [ ]:
# 14) Save class summary

class_folder_df.to_csv("/kaggle/working/class_folder_summary.csv", index=False)
print("Saved: /kaggle/working/class_folder_summary.csv")

In [ ]:
# 15) Detect train / test / validation folders

split_mask = folder_df["folder"].str.lower().str.contains(
    r'(^|/)(train|test|val|validation)(/|$)',
    regex=True
)

split_df = folder_df[split_mask].copy()

split_df[["folder", "depth", "subfolders", "is_leaf"]].head(50)

In [ ]:
# 16) Save split folder summary

split_df.to_csv("/kaggle/working/split_folder_summary.csv", index=False)
print("Saved: /kaggle/working/split_folder_summary.csv")

In [ ]:
# 17) Heuristic check for signer-like folder names

def signer_hint(name):
    n = str(name).lower()
    patterns = [
        r'^s\d+$',
        r'^p\d+$',
        r'^u\d+$',
        r'^user\d+$',
        r'^subject\d+$',
        r'^signer\d+$'
    ]
    if any(re.search(p, n) for p in patterns):
        return True

    keywords = ["signer", "subject", "speaker", "user", "participant"]
    return any(k in n for k in keywords)

folder_df["signer_hint"] = folder_df["name"].apply(signer_hint)

signer_df = folder_df[folder_df["signer_hint"]].copy()
signer_df[["folder", "name", "depth", "subfolders", "is_leaf"]].head(50)

In [ ]:
# 18) Save signer-like folder summary

signer_df.to_csv("/kaggle/working/signer_folder_summary.csv", index=False)
print("Saved: /kaggle/working/signer_folder_summary.csv")

In [ ]:
# 19) Total files

total_files = 0

for root, dirs, files in os.walk(DATASET_PATH):

    total_files += len(files)

print(total_files)

In [ ]:
# 20) Total folders

total_folders = 0

for root, dirs, files in os.walk(DATASET_PATH):

    total_folders += len(dirs)

print(total_folders)

In [ ]:
# 21) Collect All Video Paths

VIDEO_EXTENSIONS = (
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
)

video_paths = []

for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:

        if file.lower().endswith(VIDEO_EXTENSIONS):

            video_paths.append(
                os.path.join(root, file)
            )

print("Total Videos :", len(video_paths))

In [ ]:
# 22) Create Metadata Extraction Function

def get_video_metadata(video_path):

    cap = cv2.VideoCapture(video_path)

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    duration = 0

    if fps > 0:

        duration = frame_count / fps

    cap.release()

    return (
        frame_count,
        fps,
        width,
        height,
        duration
    )

In [ ]:
# 23) Extract Metadata for Every Video

metadata = []

for video in tqdm(video_paths):

    frame_count, fps, width, height, duration = get_video_metadata(video)

    metadata.append({

        "filepath": video,

        "filename": os.path.basename(video),

        "class": os.path.basename(os.path.dirname(video)),

        "frame_count": frame_count,

        "fps": fps,

        "width": width,

        "height": height,

        "duration": duration

    })

In [ ]:
# 24) Create Metadata DataFrame

metadata_df = pd.DataFrame(metadata)

metadata_df.head()

In [ ]:
# 25) Save Metadata CSV

metadata_df.to_csv(

    "/kaggle/working/signbd_metadata.csv",

    index=False

)

print("Metadata saved.")

In [ ]:
# 26) Dataset Summary

print("="*50)

print("Dataset Summary")

print("="*50)

print("Total Videos :", len(metadata_df))

print("Total Classes :", metadata_df["class"].nunique())

print("Average Duration :", metadata_df["duration"].mean())

print("Average FPS :", metadata_df["fps"].mean())

print("Average Frame Count :", metadata_df["frame_count"].mean())

print("Average Width :", metadata_df["width"].mean())

print("Average Height :", metadata_df["height"].mean())

print("="*50)

In [ ]:
# 27) Videos per Class

class_distribution = (

    metadata_df["class"]

    .value_counts()

    .sort_index()

)

class_distribution

In [ ]:
# 28) Class Distribution Plot

plt.figure(figsize=(20,6))

class_distribution.plot.bar()

plt.xlabel("Word Class")

plt.ylabel("Number of Videos")

plt.title("Videos per Class")

plt.tight_layout()

plt.show()

In [ ]:
# 29) Duration Histogram

plt.figure(figsize=(10,5))

plt.hist(

    metadata_df["duration"],

    bins=30

)

plt.xlabel("Duration (Seconds)")

plt.ylabel("Number of Videos")

plt.title("Video Duration Distribution")

plt.show()

In [ ]:
# 30) FPS Histogram

plt.figure(figsize=(10,5))

plt.hist(

    metadata_df["fps"],

    bins=20

)

plt.xlabel("FPS")

plt.ylabel("Number of Videos")

plt.title("FPS Distribution")

plt.show()

In [ ]:
# 31) Frame Count Histogram

plt.figure(figsize=(10,5))

plt.hist(

    metadata_df["frame_count"],

    bins=30

)

plt.xlabel("Frames")

plt.ylabel("Videos")

plt.title("Frame Count Distribution")

plt.show()

In [ ]:
# 32) Resolution Statistics

resolution_df = (

    metadata_df

    .groupby(

        ["width","height"]

    )

    .size()

    .reset_index(name="count")

)

resolution_df

In [ ]:
# 33) Resolution Distribution Plot

plt.figure(figsize=(10,5))

labels = (

    resolution_df["width"]

    .astype(str)

    + "×"

    + resolution_df["height"]

    .astype(str)

)

plt.bar(

    labels,

    resolution_df["count"]

)

plt.xticks(

    rotation=45

)

plt.ylabel("Videos")

plt.xlabel("Resolution")

plt.title("Resolution Distribution")

plt.tight_layout()

plt.show()

In [ ]:
# 34) Top 20 Largest Videos (by frame count)

metadata_df.sort_values(

    "frame_count",

    ascending=False

).head(20)

In [ ]:
# 35) Top 20 Longest Videos

metadata_df.sort_values(

    "duration",

    ascending=False

).head(20)

In [ ]:
# 36) Statistical Summary

metadata_df.describe()

In [ ]:
# 37) Export Final Statistics

metadata_df.describe().to_csv(

    "/kaggle/working/statistical_summary.csv"

)

In [ ]:
# 38) Generate Final Report

print("="*60)

print("SIGNBD DATASET REPORT")

print("="*60)

print("Total Videos :", len(metadata_df))

print("Total Classes :", metadata_df["class"].nunique())

print("Videos/Class :", len(metadata_df)/metadata_df["class"].nunique())

print()

print("Average Duration :", metadata_df["duration"].mean())

print("Average FPS :", metadata_df["fps"].mean())

print("Average Frames :", metadata_df["frame_count"].mean())

print("Resolution Types :")

print(

resolution_df

)

print("="*60)